# 06.7 — The confidence P-controller

This is the single **highest-risk port in the codebase** — and it looks deceptively small. The MATLAB calls it an *"Autonomous Equilibrium Controller"* and a *"PD-controller"*, names that suggest a sophisticated control system. Read the actual arithmetic and it's a **pure P-controller** (proportional only — no derivative, no integral term): three lines that nudge a scalar `beta` toward keeping the model's average confidence at 0.5. This notebook dissects it, simulates it live, and shows why the name is a trap.

This notebook covers:

* What a proportional controller is (the thermostat analogy).
* The exact update rule — `beta ← beta·(1 + (0.5 − mean_confidence))`, clamped.
* Why it's a P-controller, *not* the PD-controller its name claims.
* The closed feedback loop: how `beta` scales the loss to hold confidence at 0.5.

**Prerequisites:** [06.6 confidence routing](06.6_confidence_routing.ipynb), [06.4 EMA priors](06.4_the_ema_prior_normalization_deep_dive.ipynb).

## Section 1 — What MATLAB does

`cgg_getConfidenceLossInformation.m` (lines 60-75) updates a running `Confidence_Beta`:

```matlab
% "Autonomous Equilibrium Controller" (its name in the code)
diff    = Beta_Target - mean(TotalConfidence, "all");   % 0.5 − current confidence
newBeta = Beta .* (1 + diff .* Beta_DifferenceRate);    % proportional nudge
newBeta = min(max(newBeta, Beta_Min), Beta_Max);        % clamp to [0.1, 10]
```

`Beta` then *scales the confidence loss* (`cgg_getLossInformation`) — a larger `beta` puts more weight on the confidence term, pushing the model to change its confidence; a smaller `beta` eases off. The constants: `Beta_Target = 0.5`, `Beta_DifferenceRate = 1.0`, `Beta_Min = 0.1`, `Beta_Max = 10.0`. That's the entire controller. The grand name hides three lines of proportional feedback — the port's comments say so explicitly.

## Section 2 — The Python concepts you need

### 2.1 — A proportional controller, in one breath

A **P-controller** watches a *measured value*, compares it to a *setpoint*, and applies a correction *proportional to the gap*. A home thermostat is the canonical example: the further the room is below your target temperature, the harder it drives the heater. No memory of past error (that's the "I", integral), no reaction to the rate of change (that's the "D", derivative) — just "how far off are we right now, times a gain."

Here the measured value is the batch's **mean TotalConfidence** ([06.6](06.6_confidence_routing.ipynb)), the setpoint is **0.5**, and the correction adjusts **beta** (which scales the confidence loss). The gap is `0.5 − mean_confidence`.

### 2.2 — The exact update, spelled out

In [ ]:
import torch

BETA_TARGET = 0.5      # setpoint: we want mean confidence ≈ 0.5 (calibrated)
BETA_RATE   = 1.0      # proportional gain
BETA_MIN, BETA_MAX = 0.1, 10.0

def update_beta(beta_prev, mean_confidence):
    diff = BETA_TARGET - mean_confidence           # the gap (signed)
    new_beta = beta_prev * (1.0 + diff * BETA_RATE)  # proportional, MULTIPLICATIVE nudge
    return float(min(max(new_beta, BETA_MIN), BETA_MAX))   # clamp

# One step, over-confident model (mean=0.8 > target 0.5):
print("over-confident (0.8): beta 1.00 →", round(update_beta(1.0, 0.8), 3), "(shrinks → ease off)")
# One step, under-confident model (mean=0.3 < target 0.5):
print("under-confident (0.3): beta 1.00 →", round(update_beta(1.0, 0.3), 3), "(grows → push harder)")

Read the direction carefully:

* Model is **over-confident** (mean 0.8 > 0.5) → `diff = 0.5 − 0.8 = −0.3` → `beta × 0.7` → beta **shrinks** → confidence loss weighted *down* → less pressure → confidence eases back toward 0.5.
* Model is **under-confident** (mean 0.3 < 0.5) → `diff = +0.2` → `beta × 1.2` → beta **grows** → confidence loss weighted *up* → more pressure → confidence rises toward 0.5.

It's **negative feedback**: whichever way confidence strays, beta moves to pull it back. And the nudge is **multiplicative** (`beta × (1 + diff)`), not additive (`beta + diff`) — so the correction is proportional to beta's *current* size, and beta can't go negative from a single step.

### 2.3 — Why 0.5? Calibration

A confidence of 0.5 means "I'm right about half the time" — the maximum-entropy, least-committed state. Driving *mean* confidence to 0.5 doesn't force *every* trial to 0.5; it keeps the *distribution* centered so the model spreads confidence meaningfully (some trials 0.9, some 0.1) instead of collapsing to all-certain or all-unsure ([06.6 §5](06.6_confidence_routing.ipynb)). The controller is a calibration regularizer.

### 2.4 — Watch it converge (live simulation)

In [ ]:
# Closed loop: beta scales the confidence loss, which pushes confidence toward 0.5.
# Simple model of that feedback: confidence responds to beta (higher beta → the loss
# pulls confidence DOWN toward target harder). We just show the controller settling.
import matplotlib.pyplot as plt

def simulate(start_conf, steps=25):
    beta, conf = 1.0, start_conf
    betas, confs = [beta], [conf]
    for _ in range(steps):
        beta = update_beta(beta, conf)
        # Toy plant: confidence drifts toward 0.5 at a rate modulated by beta.
        conf = conf + 0.15 * beta * (BETA_TARGET - conf) / max(beta, 1.0)
        betas.append(beta); confs.append(conf)
    return betas, confs

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
for start in (0.05, 0.95):
    betas, confs = simulate(start)
    ax1.plot(confs, marker=".", label=f"start conf={start}")
    ax2.plot(betas, marker=".", label=f"start conf={start}")
ax1.axhline(0.5, ls="--", color="gray"); ax1.set_title("mean confidence → 0.5 setpoint")
ax1.set_xlabel("controller step"); ax1.set_ylabel("mean confidence"); ax1.legend()
ax2.axhline(1.0, ls="--", color="gray"); ax2.set_title("beta (the controller output)")
ax2.set_xlabel("controller step"); ax2.set_ylabel("beta"); ax2.legend()
plt.tight_layout(); plt.show()
print("Both starting points (0.05 and 0.95) are driven toward the 0.5 setpoint.")

The left panel shows the plant (mean confidence) converging to the 0.5 setpoint from both extremes; the right shows beta — the controller's output — settling as the gap closes. (The toy plant is illustrative; the real "plant" is the neural network responding to a beta-weighted loss over many batches. The *controller* code, though, is exactly the production `update_beta`.)

### 2.5 — The clamp is a safety rail

In [ ]:
# Without the [0.1, 10] clamp, a persistent gap compounds multiplicatively and beta runs away.
def update_unclamped(beta_prev, mean_confidence):
    return beta_prev * (1.0 + (BETA_TARGET - mean_confidence) * BETA_RATE)

beta_c, beta_u = 1.0, 1.0
for _ in range(30):                         # a stuck, always-zero-confidence model
    beta_c = update_beta(beta_c, 0.0)       # clamped
    beta_u = update_unclamped(beta_u, 0.0)  # unclamped
print(f"clamped   beta after 30 steps: {beta_c:.2f}   (pinned at max 10.0)")
print(f"unclamped beta after 30 steps: {beta_u:.2e}  (exploded)")

If the model gets stuck (say confidence pinned at 0), the multiplicative update compounds — `1.5³⁰ ≈ 190,000×`. The `[0.1, 10]` clamp caps beta so a pathological batch can't blow the confidence loss up and destabilize the whole multi-objective sum ([06.1](06.1_multi_task_losses_overview.ipynb)). It's the same defensive instinct as gradient clipping ([05.3](../05_training_loop/05.3_gradient_clipping.ipynb)).

## Section 3 — The `neural_data_decoding` implementation

`_update_confidence_beta` — the entire controller, three lines of arithmetic:

In [ ]:
from pathlib import Path
src = Path("../../src/neural_data_decoding/training/losses/confidence.py").read_text().splitlines()
i = next(j for j, line in enumerate(src) if "diff = CONFIDENCE_BETA_TARGET" in line)
for k in range(i, i + 4):
    print(f"{k + 1:4} | {src[k]}")

Things to spot:

* `diff = CONFIDENCE_BETA_TARGET - total_batch_mean` — the gap. `new_beta = beta_prev * (1.0 + diff * RATE)` — the proportional, multiplicative nudge. `.clamp(MIN, MAX)` — the safety rail. `.detach()` — beta is *controller state*, not a gradient path ([02.5 §2.4](../02_numpy_and_pytorch_basics/02.5_autograd_basics.ipynb)); the network must not backprop into the controller.
* **The name is a trap.** The function docstring says it outright: MATLAB calls it *"Autonomous Equilibrium Controller"* / *"PD-controller"*, but there is no D (derivative) term and no I (integral) term — it's pure P. This is the curriculum's recurring lesson ([06.4](06.4_the_ema_prior_normalization_deep_dive.ipynb), [03.5](../03_data_pipeline/03.5_normalization_strategies.ipynb)): **read the code, not the label.** Trusting the "PD" name would send you hunting for a derivative term that isn't there.
* The batch mean uses the **undropped** TotalConfidence (§ the routing kernel, [06.6](06.6_confidence_routing.ipynb)) — Critical Note #29 subtlety: beta measures the model's *actual* confidence, not the dropout-ablated version.
* Why it's the highest-risk port: it's tiny, it's mislabeled, its direction (multiplicative, signed gap) is easy to flip, and a sign error wouldn't crash — it would silently *destabilize* confidence instead of calibrating it. Only an empirical parity probe against the MATLAB trajectory catches that.

## Section 4 — Hands-on exercises

### Exercise 4.1 — find the equilibrium

At what mean confidence does beta stop changing? Solve it, then confirm numerically.

In [ ]:
# Reveal:
# beta is unchanged when the multiplier is 1, i.e. when diff = 0, i.e. mean_confidence = 0.5.
for mean_c in (0.3, 0.5, 0.7):
    b = update_beta(2.0, mean_c)
    print(f"  mean confidence {mean_c}: beta 2.00 → {b:.3f}  {'(fixed point!)' if abs(b-2.0)<1e-9 else ''}")
print("→ beta is stationary exactly at the 0.5 setpoint — the equilibrium the controller drives toward.")

### Exercise 4.2 — it's a P-controller, prove the absence of D

A real PD-controller's output depends on the *rate of change* of the error (the difference between consecutive gaps). Show this controller gives the SAME beta step for the same current gap, regardless of what the previous gap was (no memory of history → no derivative term).

In [ ]:
# Reveal:
# Same current confidence (0.7) reached via two different histories:
b_from_low  = update_beta(1.0, 0.7)   # previous batch was, say, 0.3
b_from_high = update_beta(1.0, 0.7)   # previous batch was, say, 0.9
print(f"beta step from current conf 0.7 (prev low):  {b_from_low:.4f}")
print(f"beta step from current conf 0.7 (prev high): {b_from_high:.4f}")
print("→ identical: the update ignores the PREVIOUS gap. A PD-controller would differ. Pure P.")

## Section 5 — Common errors

### The sign is flipped (the silent killer)

`diff = TARGET - mean`, not `mean - TARGET`. Flip it and beta *grows* when the model is over-confident, amplifying the runaway instead of damping it. It won't crash — confidence just fails to calibrate, or diverges. This is the exact failure the empirical parity probe exists to catch; don't trust the arithmetic by eye.

### Additive instead of multiplicative

`beta * (1 + diff)`, not `beta + diff`. The multiplicative form makes the correction proportional to beta's current magnitude and keeps beta positive. An additive port drifts and can go negative, inverting the loss weighting.

### Chasing the "PD" / "derivative" term

There isn't one. The name promises a derivative (and integral) term the code never implements. Time spent looking for it is time wasted — the docstring flags this explicitly. Read the three lines, not the label.

### Beta backpropagated

Beta is `.detach()`-ed — it's controller state, like the EMA priors ([06.4](06.4_the_ema_prior_normalization_deep_dive.ipynb)) and confidence history ([06.6](06.6_confidence_routing.ipynb)). If it carried gradient, the network could game the controller by manipulating beta instead of its actual confidence.

### Using dropped confidence for the beta mean

The controller measures the model's *real* mean confidence (undropped) — Critical Note #29. Feeding it the dropout-ablated confidence measures the wrong quantity and mis-calibrates the setpoint.

### Beta pinned at a clamp bound

If beta sits at 0.1 or 10.0 for many steps, the model is persistently mis-confident (stuck under/over the setpoint) — the controller is saturated. That's a signal to inspect the confidence heads or weights, not a controller bug; the clamp is doing its job.

## Section 6 — Further reading

- [PID controller (Wikipedia)](https://en.wikipedia.org/wiki/PID_controller) — the P/I/D decomposition this notebook's "it's only P" argument rests on.
- [`src/neural_data_decoding/training/losses/confidence.py`](../../src/neural_data_decoding/training/losses/confidence.py) — `_update_confidence_beta` and the routing kernel that calls it.
- [06.6 confidence routing](06.6_confidence_routing.ipynb) — where the mean TotalConfidence the controller reads comes from.

Next up: **[06.8 L2 inside the loss kernel](06.8_l2_inside_the_loss_kernel.ipynb)** — why the project computes L2 by hand instead of using the optimizer's `weight_decay`.